In [1]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, classification_report
import numpy as np
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
from pytorch_tabnet.tab_model import TabNetClassifier

In [2]:
df = pd.read_csv('foods_health_scores_allergens.csv')
df

,product_name,brands,categories,ingredients,nutriscore_grade,nova_group,ecoscore_grade,allergens,energy_kcal,fat_100g,...,proteins_100g,salt_100g,sodium_100g,contains_gluten,contains_dairy,contains_nuts,contains_soy,contains_eggs,contains_fish,food_type
0,Sidi Ali,سيدي علي,"en:beverages-and-beverages-preparations, en:be...",OBD1 999 999 1112606 266963207 mb,A,NaN,NOT-APPLICABLE,NaN,0.0,0.0,...,0.0,0.000000,0.000000,False,False,False,False,False,False,Branded/Packaged
1,Perly,Perly,"en:dairies, en:fermented-foods, en:fermented-m...","milk cream, cream, sugar, banana, bacteria",UNKNOWN,3.0,B,"en:banana, en:milk",97.0,3.0,...,8.0,NaN,NaN,False,True,False,False,False,False,Branded/Packaged
2,Sidi Ali,Sidi Ali,"en:beverages-and-beverages-preparations, en:be...","Sodium, Calcium, Magnésium, Potassium, Bicarbo...",A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,...,NaN,0.065000,0.026000,False,False,False,False,False,False,Branded/Packaged
3,Eau minérale naturelle,sidi ali,"en:beverages-and-beverages-preparations, en:be...",100% mineral water,A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,...,NaN,0.065000,0.026000,False,False,False,False,False,False,Branded/Packaged
4,اكوافينا,AQUAFINA,"en:beverages-and-beverages-preparations, en:be...",ouverture et avant le : Voir bouteille. après ...,A,NaN,NOT-APPLICABLE,NaN,0.0,0.0,...,0.0,0.000508,0.000203,False,False,False,False,False,False,Branded/Packaged
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4992,Crème fraîche gastronomique,Président,"en:dairies, en:fermented-foods, en:fermented-m...","_Crème_ (origine France), _ferments lactiques_",D,3.0,A,en:milk,291.0,30.0,...,2.3,0.070000,0.028000,False,True,False,False,False,False,Branded/Packaged
4993,Noix de cajou grillées sans sel,Maître Prunille,"en:plant-based-foods-and-beverages, en:plant-b...",Noix de cajou,B,1.0,E,en:nuts,621.0,47.0,...,21.0,0.020000,0.008000,False,False,True,False,False,False,Branded/Packaged
4994,Cacao puro en polvo desgrasado,Valor,"en:cocoa-and-its-products, en:cocoa-and-chocol...","Cacao desgrasado en polvo, correctores de acid...",C,1.0,F,NaN,375.0,16.0,...,26.0,0.030000,0.012000,False,False,False,False,False,False,Branded/Packaged
4995,Sablés Myrtilles Germes de riz,Gerblé,"en:snacks, en:sweet-snacks, en:biscuits-and-cakes","Farine de blé 58,2%, huile de colza, sucre de ...",A,4.0,C,"en:gluten, en:milk",54.0,2.0,...,0.9,0.050000,0.020000,True,True,False,False,False,False,Branded/Packaged


In [3]:
df["food_type"].value_counts()

food_type
Branded/Packaged    4997
Name: count, dtype: int64

In [4]:
df = df.drop(['product_name', 'brands', 'categories', 'ingredients', 'allergens', 'food_type'], axis = 1)

In [5]:
cols = [
    'contains_gluten',
    'contains_dairy',
    'contains_nuts',
    'contains_soy',
    'contains_eggs',
    'contains_fish'
]

df['Y'] = df[cols].any(axis=1)

In [6]:
df['Y'] = df['Y'].astype(int)

In [7]:
df["Y"]

0       0
1       1
2       0
3       0
4       0
       ..
4992    1
4993    1
4994    0
4995    1
4996    1
Name: Y, Length: 4997, dtype: int64

In [8]:
df = df.drop(cols, axis = 1)

In [9]:
df

,nutriscore_grade,nova_group,ecoscore_grade,energy_kcal,fat_100g,saturated_fat_100g,carbs_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,Y
0,A,NaN,NOT-APPLICABLE,0.0,0.0,0.0,4.2,1.4,0.0,0.0,0.000000,0.000000,0
1,UNKNOWN,3.0,B,97.0,3.0,NaN,9.4,NaN,NaN,8.0,NaN,NaN,1
2,A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065000,0.026000,0
3,A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065000,0.026000,0
4,A,NaN,NOT-APPLICABLE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000508,0.000203,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4992,D,3.0,A,291.0,30.0,22.0,2.9,2.9,NaN,2.3,0.070000,0.028000,1
4993,B,1.0,E,621.0,47.0,8.7,25.0,6.2,4.5,21.0,0.020000,0.008000,1
4994,C,1.0,F,375.0,16.0,10.0,16.0,0.7,32.0,26.0,0.030000,0.012000,0
4995,A,4.0,C,54.0,2.0,0.2,7.9,1.9,0.5,0.9,0.050000,0.020000,1


In [10]:
df.columns

Index(['nutriscore_grade', 'nova_group', 'ecoscore_grade', 'energy_kcal',
       'fat_100g', 'saturated_fat_100g', 'carbs_100g', 'sugars_100g',
       'fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g', 'Y'],
      dtype='str')

In [11]:
y = df["Y"]
X = df.drop("Y", axis = 1)

In [12]:
y.value_counts(normalize = True)

Y
1    0.621573
0    0.378427
Name: proportion, dtype: float64

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        
    random_state=5831,      
    stratify=y           
)

In [33]:
baseline_recall = (y_test == 1).mean()
print("Baseline recall:", baseline_recall)

Baseline recall: 0.622


In [14]:
# ---- define your columns ----
num_cols = ['energy_kcal','fat_100g', 'saturated_fat_100g', 'carbs_100g', 'sugars_100g','fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g']     
cat_cols = ['nutriscore_grade', 'nova_group', 'ecoscore_grade']     

# ---- numeric pipeline ----
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

# ---- categorical pipeline ----
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# ---- combine ----
preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# ---- full pipeline ----
pipe = Pipeline([
    ("preprocess", preprocess),
    ("lda", LinearDiscriminantAnalysis())
])

In [15]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [16]:
recall = recall_score(y_test, y_pred, pos_label=1)
print("Recall:", recall)

Recall: 0.9356913183279743


In [17]:
result = permutation_importance(
    pipe, X_test, y_test,
    n_repeats=10,
    random_state=5831,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values(by="importance", ascending=False)

print(importance.head(10))

               feature  importance
0     nutriscore_grade      0.0633
2       ecoscore_grade      0.0580
1           nova_group      0.0323
3          energy_kcal      0.0314
9        proteins_100g      0.0276
4             fat_100g      0.0185
5   saturated_fat_100g      0.0134
7          sugars_100g      0.0084
11         sodium_100g      0.0034
8           fiber_100g      0.0013


In [18]:
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("qda", QuadraticDiscriminantAnalysis(reg_param=0.1))
])

In [19]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [20]:
recall = recall_score(y_test, y_pred, pos_label=1)
print("Recall:", recall)

Recall: 0.9758842443729904


In [21]:
result = permutation_importance(
    pipe, X_test, y_test,
    n_repeats=10,
    random_state=5831,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values(by="importance", ascending=False)

print(importance.head(10))

               feature  importance
2       ecoscore_grade      0.0465
3          energy_kcal      0.0141
0     nutriscore_grade      0.0121
4             fat_100g      0.0091
5   saturated_fat_100g      0.0053
1           nova_group      0.0025
11         sodium_100g      0.0022
8           fiber_100g      0.0018
9        proteins_100g      0.0016
7          sugars_100g      0.0013


In [22]:
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False))
])

preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("lda", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))
])

In [23]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [24]:
recall = recall_score(y_test, y_pred, pos_label=1)
print("Recall:", recall)

Recall: 0.932475884244373


In [25]:
result = permutation_importance(
    pipe, X_test, y_test,
    n_repeats=10,
    random_state=5831,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values(by="importance", ascending=False)

print(importance.head(10))

               feature  importance
0     nutriscore_grade      0.0640
2       ecoscore_grade      0.0495
1           nova_group      0.0215
9        proteins_100g      0.0206
3          energy_kcal      0.0119
4             fat_100g      0.0046
7          sugars_100g      0.0035
10           salt_100g      0.0024
5   saturated_fat_100g      0.0020
11         sodium_100g      0.0019


In [26]:
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5))
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("xgb", XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=5831,
        eval_metric="logloss"
    ))
])

In [27]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [28]:
recall = recall_score(y_test, y_pred, pos_label=1)
print("Recall:", recall)

Recall: 0.9308681672025724


In [29]:
result = permutation_importance(
    pipe, X_test, y_test,
    n_repeats=10,
    random_state=5831,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values(by="importance", ascending=False)

print(importance.head(10))

               feature  importance
9        proteins_100g      0.1446
4             fat_100g      0.0482
7          sugars_100g      0.0473
3          energy_kcal      0.0408
6           carbs_100g      0.0369
10           salt_100g      0.0358
2       ecoscore_grade      0.0315
5   saturated_fat_100g      0.0315
8           fiber_100g      0.0290
11         sodium_100g      0.0246


In [30]:
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5))
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("lgbm", LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=5831
    ))
])

In [31]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

[LightGBM] [Info] Number of positive: 2484, number of negative: 1513
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000386 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2304
[LightGBM] [Info] Number of data points in the train set: 3997, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.621466 -> initscore=0.495776
[LightGBM] [Info] Start training from score 0.495776


/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [32]:
recall = recall_score(y_test, y_pred, pos_label=1)
print("Recall:", recall)

Recall: 0.9244372990353698


In [32]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    result = permutation_importance(
    pipe, X_test, y_test,
    n_repeats=10,
    random_state=5831,
    n_jobs=-1
    )

    importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
    }).sort_values(by="importance", ascending=False)

    print(importance.head(10)) 

               feature  importance
9        proteins_100g      0.1626
4             fat_100g      0.0579
7          sugars_100g      0.0494
3          energy_kcal      0.0476
6           carbs_100g      0.0370
5   saturated_fat_100g      0.0367
10           salt_100g      0.0356
2       ecoscore_grade      0.0326
8           fiber_100g      0.0313
11         sodium_100g      0.0269


In [41]:
results = []

for n_d in [16, 32, 64]:
    for n_steps in [3, 5, 7]:
        for lr in [0.01, 0.02, 0.05]:
            model = TabNetClassifier(
                seed=5831,
                n_d=n_d,
                n_a=n_d,
                n_steps=n_steps,
                gamma=1.5,
                lambda_sparse=1e-4,
                optimizer_params=dict(lr=lr),
                mask_type="entmax"
            )

            model.fit(
                X_train_sub_trans,
                y_train_sub_np,
                eval_set=[(X_valid_trans, y_valid_np)],
                eval_name=["valid"],
                eval_metric=["auc"],
                max_epochs=200,
                patience=20,
                batch_size=256,
                virtual_batch_size=64,
                num_workers=0,
                drop_last=False
            )

            y_valid_prob = model.predict_proba(X_valid_trans)[:, 1]

            best_recall = -1
            best_t = None

            for t in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
                y_valid_pred_t = (y_valid_prob >= t).astype(int)
                score = recall_score(y_valid, y_valid_pred_t, pos_label=1)

                if score > best_recall:
                    best_recall = score
                    best_t = t

            results.append({
                "n_d": n_d,
                "n_steps": n_steps,
                "lr": lr,
                "best_threshold": best_t,
                "valid_pos_recall": best_recall
            })

results_df = pd.DataFrame(results).sort_values("valid_pos_recall", ascending=False)
print(results_df.head(10))

/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.72676 | valid_auc: 0.6651  |  0:00:00s
epoch 1  | loss: 0.59138 | valid_auc: 0.69843 |  0:00:00s
epoch 2  | loss: 0.54902 | valid_auc: 0.73085 |  0:00:00s
epoch 3  | loss: 0.52586 | valid_auc: 0.74296 |  0:00:01s
epoch 4  | loss: 0.50635 | valid_auc: 0.75932 |  0:00:01s
epoch 5  | loss: 0.4915  | valid_auc: 0.75793 |  0:00:01s
epoch 6  | loss: 0.48501 | valid_auc: 0.77495 |  0:00:02s
epoch 7  | loss: 0.48032 | valid_auc: 0.78387 |  0:00:02s
epoch 8  | loss: 0.47285 | valid_auc: 0.78941 |  0:00:02s
epoch 9  | loss: 0.46182 | valid_auc: 0.7927  |  0:00:02s
epoch 10 | loss: 0.46286 | valid_auc: 0.79399 |  0:00:03s
epoch 11 | loss: 0.45393 | valid_auc: 0.79873 |  0:00:03s
epoch 12 | loss: 0.45033 | valid_auc: 0.80524 |  0:00:03s
epoch 13 | loss: 0.44073 | valid_auc: 0.80584 |  0:00:04s
epoch 14 | loss: 0.45177 | valid_auc: 0.80975 |  0:00:04s
epoch 15 | loss: 0.43759 | valid_auc: 0.81762 |  0:00:04s
epoch 16 | loss: 0.43901 | valid_auc: 0.81851 |  0:00:05s
epoch 17 | los

/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.70976 | valid_auc: 0.64712 |  0:00:00s
epoch 1  | loss: 0.60062 | valid_auc: 0.71919 |  0:00:00s
epoch 2  | loss: 0.5469  | valid_auc: 0.73184 |  0:00:01s
epoch 3  | loss: 0.51922 | valid_auc: 0.76323 |  0:00:01s
epoch 4  | loss: 0.50468 | valid_auc: 0.76796 |  0:00:01s
epoch 5  | loss: 0.50108 | valid_auc: 0.77866 |  0:00:02s
epoch 6  | loss: 0.4978  | valid_auc: 0.78391 |  0:00:02s
epoch 7  | loss: 0.47931 | valid_auc: 0.79912 |  0:00:02s
epoch 8  | loss: 0.47412 | valid_auc: 0.80503 |  0:00:02s
epoch 9  | loss: 0.46495 | valid_auc: 0.8146  |  0:00:03s
epoch 10 | loss: 0.45987 | valid_auc: 0.82166 |  0:00:03s
epoch 11 | loss: 0.44156 | valid_auc: 0.82368 |  0:00:03s
epoch 12 | loss: 0.44181 | valid_auc: 0.81905 |  0:00:04s
epoch 13 | loss: 0.43428 | valid_auc: 0.81649 |  0:00:04s
epoch 14 | loss: 0.43956 | valid_auc: 0.81735 |  0:00:05s
epoch 15 | loss: 0.43855 | valid_auc: 0.81275 |  0:00:05s
epoch 16 | loss: 0.4184  | valid_auc: 0.81708 |  0:00:05s
epoch 17 | los

/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


...

/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.77211 | valid_auc: 0.59814 |  0:00:00s
epoch 1  | loss: 0.60362 | valid_auc: 0.6939  |  0:00:00s
epoch 2  | loss: 0.58055 | valid_auc: 0.6895  |  0:00:01s
epoch 3  | loss: 0.55737 | valid_auc: 0.69383 |  0:00:01s
epoch 4  | loss: 0.55631 | valid_auc: 0.68583 |  0:00:02s
epoch 5  | loss: 0.54035 | valid_auc: 0.71251 |  0:00:02s
epoch 6  | loss: 0.53274 | valid_auc: 0.73674 |  0:00:03s
epoch 7  | loss: 0.51819 | valid_auc: 0.76843 |  0:00:03s
epoch 8  | loss: 0.51394 | valid_auc: 0.77369 |  0:00:04s
epoch 9  | loss: 0.50687 | valid_auc: 0.76897 |  0:00:04s
epoch 10 | loss: 0.51628 | valid_auc: 0.76443 |  0:00:05s
epoch 11 | loss: 0.50622 | valid_auc: 0.7776  |  0:00:05s
epoch 12 | loss: 0.50041 | valid_auc: 0.7631  |  0:00:06s
epoch 13 | loss: 0.50997 | valid_auc: 0.77561 |  0:00:06s
epoch 14 | loss: 0.49675 | valid_auc: 0.75751 |  0:00:06s
epoch 15 | loss: 0.48819 | valid_auc: 0.76445 |  0:00:07s
epoch 16 | loss: 0.48467 | valid_auc: 0.77424 |  0:00:07s
epoch 17 | los

/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.90997 | valid_auc: 0.58057 |  0:00:00s
epoch 1  | loss: 0.67947 | valid_auc: 0.5866  |  0:00:01s
epoch 2  | loss: 0.63946 | valid_auc: 0.65811 |  0:00:02s
epoch 3  | loss: 0.61302 | valid_auc: 0.66036 |  0:00:02s
epoch 4  | loss: 0.60643 | valid_auc: 0.66794 |  0:00:03s
epoch 5  | loss: 0.58641 | valid_auc: 0.69891 |  0:00:04s
epoch 6  | loss: 0.58961 | valid_auc: 0.72152 |  0:00:04s
epoch 7  | loss: 0.57871 | valid_auc: 0.75869 |  0:00:05s
epoch 8  | loss: 0.56567 | valid_auc: 0.72763 |  0:00:06s
epoch 9  | loss: 0.56938 | valid_auc: 0.69297 |  0:00:06s
epoch 10 | loss: 0.56374 | valid_auc: 0.68767 |  0:00:07s
epoch 11 | loss: 0.56058 | valid_auc: 0.70536 |  0:00:08s
epoch 12 | loss: 0.56041 | valid_auc: 0.70535 |  0:00:08s
epoch 13 | loss: 0.55057 | valid_auc: 0.71344 |  0:00:09s
epoch 14 | loss: 0.57826 | valid_auc: 0.72722 |  0:00:09s
epoch 15 | loss: 0.55089 | valid_auc: 0.74263 |  0:00:10s
epoch 16 | loss: 0.54396 | valid_auc: 0.75663 |  0:00:11s
epoch 17 | los

KeyboardInterrupt: 

In [37]:
y_prob = tabnet.predict_proba(X_valid_trans)[:, 1]

for t in [0.5, 0.4, 0.3, 0.2]:
    y_pred_t = (y_prob >= t).astype(int)
    score = recall_score(y_valid, y_pred_t, pos_label=1)
    print(f"threshold={t:.2f}, positive recall={score:.4f}")

threshold=0.50, positive recall=0.9356
threshold=0.40, positive recall=0.9618
threshold=0.30, positive recall=0.9759
threshold=0.20, positive recall=0.9839
